# 00 — Setup e prima decisione
Un **decision model** legge uno *stato* più domande le cui risposte sono dichiarate in anticipo, e restituisce una probabilità per ogni risposta invece di testo generato.

Tutto qui è **libero/aperto**: dati pubblici, modelli open, librerie open-source. Il `MockClient` offline non richiede chiavi né rete; il backend predefinito è locale.


In [ ]:
from decision_qa import load_config, MockClient, SystemOneClient
from decision_qa import NoulQuestion, ChoiceQuestion, ScoreQuestion

cfg = load_config()
print('backend:', cfg.backend, '| endpoint:', cfg.endpoint, '| model:', cfg.model)


## Prima chiamata
Usiamo `MockClient` così il notebook funziona sempre, gratis, anche senza server. Sostituisci con `SystemOneClient(cfg)` quando un endpoint locale gratuito (Ollama / llama.cpp) è attivo.


In [ ]:
client = MockClient(seed=42)  # free, deterministic, offline

state = {
    'context': 'A trial of 240 patients reports a 42% reduction in headache frequency.',
    'question': 'Is the treatment effective?',
}
questions = [
    NoulQuestion(label='supported', statement='The treatment is effective.', gold=True),
    ChoiceQuestion(label='verdict', options=['effective', 'ineffective', 'unclear'], gold_index=0),
    ScoreQuestion(label='evidence', rubric='Rate the strength of evidence.',
                  levels=['weak', 'moderate', 'strong'], gold_index=2),
]

for result in client.decision(state, questions):
    print(result.question_label, {k: round(v, 3) for k, v in result.probabilities.items()})


Ogni risultato è una **distribuzione sulle risposte dichiarate**, non una frase generata. Le etichette gold servono solo per didattica/valutazione; in una richiesta reale si omettono.


In [ ]:
# Exercise: change the state and watch the probabilities move.
weak_state = {'context': 'A 5-patient anecdote mentions improvement.',
              'question': 'Is the treatment effective?'}
for result in client.decision(weak_state, questions):
    print(result.question_label, {k: round(v, 3) for k, v in result.probabilities.items()})
